# Data Preprocessing

The objective of this notebook is to prepare the malaria cell image dataset for deep learning.

The preprocessing pipeline consists of:

1. Filtering valid image files
2. Creating stratified train, validation and test splits
3. Defining image transformations
4. Creating PyTorch datasets
5. Creating PyTorch dataloaders
6. Verifying the preprocessing pipeline

In [1]:
import random
from pathlib import Path
import sys

import numpy as np
import pandas as pd


import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

from sklearn.model_selection import train_test_split

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:

# Project Configuration

# Root project directory
PROJECT_DIR = Path(
    "/content/drive/MyDrive/Malaria-Cell-Classification"
)

# Add project to Python path
if str(PROJECT_DIR) not in sys.path:
    sys.path.append(str(PROJECT_DIR))

# Project directories
DATASET_DIR = PROJECT_DIR / "dataset"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
CHECKPOINTS_DIR = PROJECT_DIR / "checkpoints"
OUTPUTS_DIR = PROJECT_DIR / "outputs"

print(f"Project Directory : {PROJECT_DIR}")

Project Directory : /content/drive/MyDrive/Malaria-Cell-Classification


In [4]:
# ============================================================
# Project Modules
# ============================================================

from src import (
    MalariaDataPreprocessor,
    set_seed,
    get_device,
)


In [5]:

# Reproducibility


set_seed(42)

device = get_device()

Using GPU : Tesla T4


We will have a class for preprocessing rather than stand alone codes

MalariaDataPreprocessor
                      │

                        ├── Configuration
                          │     dataset_name
                          │     image_size
                          │     batch_size
                          │     validation_split
                          │     test_split
                          │     random_state
                          │     num_workers
                      │
                        ├── Dataset Paths
                          │     dataset_path
                          │     cell_images_path
                      │
                        ├── DataFrames
                          │     data
                          │     train_df
                          │     val_df
                          │     test_df
                      │
                        ├── Transforms
                          │     train_transform
                          │     eval_transform
                      │
                        ├── Dataset Objects
                          │     train_dataset
                          │     val_dataset
                          │     test_dataset
                      │
                        └── DataLoaders
                          train_loader
                          val_loader
                          test_loader

In [7]:
# ============================================================
# Preprocessing Configuration
# ============================================================

CONFIG = {

    "project_dir": PROJECT_DIR,

    "image_size": (128, 128),

    "batch_size": 32,

    "validation_split": 0.15,

    "test_split": 0.15,

    "random_state": 42,

    "num_workers": 0,

    "pin_memory": True,

    "rotation_degrees": 15,

    "horizontal_flip_prob": 0.5,

    "vertical_flip_prob": 0.5,

    "brightness": 0.10,

    "contrast": 0.10,

    "saturation": 0.10,
}

In [ ]:
import importlib
import src.preprocessing

importlib.reload(src.preprocessing)

<module 'src.preprocessing' from '/content/drive/MyDrive/Malaria-Cell-Classification/src/preprocessing.py'>

In [9]:
# ============================================================
# Run Preprocessing Pipeline
# ============================================================

preprocessor = MalariaDataPreprocessor(
    project_dir=CONFIG["project_dir"],
    image_size=CONFIG["image_size"],
    batch_size=CONFIG["batch_size"],
    validation_split=CONFIG["validation_split"],
    test_split=CONFIG["test_split"],
    random_state=CONFIG["random_state"],
    num_workers=CONFIG["num_workers"],
    pin_memory=CONFIG["pin_memory"],
    rotation_degrees=CONFIG["rotation_degrees"],
    horizontal_flip_prob=CONFIG["horizontal_flip_prob"],
    vertical_flip_prob=CONFIG["vertical_flip_prob"],
    brightness=CONFIG["brightness"],
    contrast=CONFIG["contrast"],
    saturation=CONFIG["saturation"],
)


preprocessor.prepare()

Dataset already extracted.

Dataset validation successful.
Dataset Location : /content/drive/MyDrive/Malaria-Cell-Classification/dataset/extracted/cell_images
Classes Found    : ['Parasitized', 'Uninfected']
Dataset Metadata
Total Images : 27558

   image_id                                           filepath  \
0         0  /content/drive/MyDrive/Malaria-Cell-Classifica...   
1         1  /content/drive/MyDrive/Malaria-Cell-Classifica...   
2         2  /content/drive/MyDrive/Malaria-Cell-Classifica...   
3         3  /content/drive/MyDrive/Malaria-Cell-Classifica...   
4         4  /content/drive/MyDrive/Malaria-Cell-Classifica...   

                                        filename   class_name  label  width  \
0  C100P61ThinF_IMG_20150918_144104_cell_162.png  Parasitized      1    142   
1  C100P61ThinF_IMG_20150918_144104_cell_163.png  Parasitized      1    148   
2  C100P61ThinF_IMG_20150918_144104_cell_164.png  Parasitized      1    139   
3  C100P61ThinF_IMG_20150918_144104_cell

In [10]:
print(f"Training Batches   : {len(preprocessor.train_loader)}")
print(f"Validation Batches : {len(preprocessor.val_loader)}")
print(f"Testing Batches    : {len(preprocessor.test_loader)}")

Training Batches   : 603
Validation Batches : 130
Testing Batches    : 130


In [11]:
images, labels, image_ids = next(iter(preprocessor.train_loader))

print("Images Shape :", images.shape)
print("Labels Shape :", labels.shape)
print("Image IDs Shape :", image_ids.shape)

Images Shape : torch.Size([32, 3, 128, 128])
Labels Shape : torch.Size([32])
Image IDs Shape : torch.Size([32])


In [12]:
from pathlib import Path

print("Artifacts")

for file in sorted((PROJECT_DIR / "artifacts").iterdir()):
    print(f" - {file.name}")

Artifacts
 - normalization.json
 - transforms.json


### **Preprocessing Summary** ###

The preprocessing pipeline automatically performed the following tasks:



 1. Located and extracted the dataset from the project directory.
 2. Validated the dataset structure and image files.
 3. Prepared the dataset metadata.
 4. Split the data into training, validation, and testing subsets using stratified sampling to preserve class balance.
 5. Computed the normalization statistics (mean and standard deviation) from the training set.
 6. Saved the computed statistics as reusable artifacts.
 7. Constructed the image transformation pipelines for both training and evaluation.
 8. Created PyTorch Dataset objects for each data split.
 9. Built optimized DataLoader objects that will be used throughout the model training process.


As part of the preprocessing stage, several project artifacts were generated. These artifacts provide reproducibility and eliminate the need to recompute preprocessing information in future notebooks.

The artifacts directory now contains the preprocessing metadata, including:

- Dataset normalization statistics (mean and standard deviation).
-  Any additional preprocessing metadata required for consistent evaluation and inference.

The preprocessing pipeline also prepared the in-memory objects that will be consumed by subsequent stages of the project:

a).  Training, validation, and testing datasets.

b).  Training, validation, and testing dataloaders.

c).  Image transformation pipelines for training and evaluation.








